## **1. Install & Import Libraries**

Install scikit-learn untuk permodelan dan imbalanced-learn untuk teknik SMOTE.

In [3]:
# Install library untuk penyeimbangan data
!pip install imbalanced-learn

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE
import joblib

## **2. Load Data & Labelling**

Kita memuat ReviewAppDiscord_Clean.csv. Dan memberikan label:

* Positif (1): Rating 4 dan 5.

* Negatif (0): Rating 1 dan 2.

* Rating 3 biasanya dibuang karena bersifat netral/ambigu.

In [4]:
df = pd.read_csv('ReviewAppDiscord_Clean.csv')
df = df.dropna(subset=['content_stemmed'])

# Pelabelan otomatis
df = df[df['score'] != 3] # Menghapus rating netral
df['label'] = df['score'].apply(lambda x: 1 if x > 3 else 0)

print(f"Distribusi Label:\n{df['label'].value_counts()}")

Distribusi Label:
label
0    6929
1    3542
Name: count, dtype: int64


## **3. TF-IDF Vectorization**

Mengubah teks menjadi matriks angka. Kita batasi max_features agar laptop tidak berat saat proses training.

In [5]:
tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['content_stemmed'])
y = df['label']

## **4. Train-Test Split**

Membagi data menjadi 80% untuk belajar (Training) dan 20% untuk ujian (Testing).

In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## **5. Handling Imbalance Data (SMOTE)**

Kita gunakan SMOTE agar jumlah data positif dan negatif menjadi seimbang. Ini penting agar model tidak "pilih kasih".

In [7]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Jumlah data setelah SMOTE: {X_train_res.shape[0]}")

Jumlah data setelah SMOTE: 11046


## **6. Model Comparison (Benchmarking)**

Kita bandingkan 4 model sekaligus untuk melihat mana yang terbaik.

In [8]:
models = {
    "Logistic Regression": LogisticRegression(),
    "SVM (Linear)": SVC(kernel='linear'),
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100)
}

for name, model in models.items():
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))


=== Logistic Regression ===
Accuracy: 0.8649
              precision    recall  f1-score   support

           0       0.92      0.88      0.90      1406
           1       0.77      0.84      0.80       689

    accuracy                           0.86      2095
   macro avg       0.84      0.86      0.85      2095
weighted avg       0.87      0.86      0.87      2095


=== SVM (Linear) ===
Accuracy: 0.8654
              precision    recall  f1-score   support

           0       0.92      0.88      0.90      1406
           1       0.77      0.84      0.80       689

    accuracy                           0.87      2095
   macro avg       0.84      0.86      0.85      2095
weighted avg       0.87      0.87      0.87      2095


=== Naive Bayes ===
Accuracy: 0.8654
              precision    recall  f1-score   support

           0       0.90      0.90      0.90      1406
           1       0.79      0.80      0.80       689

    accuracy                           0.87      2095
   ma

## **7. Save Model & Vectorizer**

Langkah terakhir adalah menyimpan model terbaik (misal SVM) dan vectorizer TF-IDF.

In [9]:
# Simpan TF-IDF dan Model (Contoh simpan SVM)
joblib.dump(tfidf, 'tfidf_discord.pkl')
joblib.dump(models["SVM (Linear)"], 'svm_model_discord.pkl')
print("Model dan TF-IDF Berhasil Disimpan!")

Model dan TF-IDF Berhasil Disimpan!


## **8. Uji Coba Model (Predicting New Reviews)**

Di tahap ini, kita akan mengetes "kecerdasan" model SVM yang sudah disimpan tadi menggunakan kalimat-kalimat yang kita buat sendiri.

In [10]:
import joblib

# 1. Load Model dan Vectorizer yang sudah disimpan tadi
loaded_tfidf = joblib.load('tfidf_discord.pkl')
loaded_model = joblib.load('svm_model_discord.pkl')

# 2. Fungsi untuk Prediksi Otomatis
def predict_sentiment(text):
    # Transformasi teks baru menggunakan TF-IDF yang sudah dilatih
    text_vector = loaded_tfidf.transform([text])

    # Melakukan prediksi
    prediction = loaded_model.predict(text_vector)

    # Mengubah angka (0/1) menjadi label teks
    result = "Positif" if prediction[0] == 1 else "Negatif"
    return result

# 3. Mari kita tes!
test_sentences = [
    "Aplikasi ini sangat membantu saya dalam berkomunikasi dengan teman game",
    "Sering banget error dan koneksi tidak stabil, kecewa sekali",
    "Fiturnya keren dan mudah digunakan oleh pemula",
    "Banyak bug pas mau login, tolong segera diperbaiki"
]

print("=== Hasil Prediksi Sentimen Kalimat Baru ===")
for sentence in test_sentences:
    sentiment = predict_sentiment(sentence)
    print(f"Kalimat: {sentence}")
    print(f"Sentimen: {sentiment}")
    print("-" * 30)

=== Hasil Prediksi Sentimen Kalimat Baru ===
Kalimat: Aplikasi ini sangat membantu saya dalam berkomunikasi dengan teman game
Sentimen: Positif
------------------------------
Kalimat: Sering banget error dan koneksi tidak stabil, kecewa sekali
Sentimen: Negatif
------------------------------
Kalimat: Fiturnya keren dan mudah digunakan oleh pemula
Sentimen: Positif
------------------------------
Kalimat: Banyak bug pas mau login, tolong segera diperbaiki
Sentimen: Negatif
------------------------------
